# Data and Model Versioning

Notebook for **Chapter 4** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch04.html

Tip: change a number, predict the output, then run the cell again.

## 4.1 Versioning Data with Content Hashes (the DVC Idea)

In [ ]:
import hashlib, json, pathlib, shutil
ROOT = pathlib.Path("/tmp/lab/dvcdemo"); shutil.rmtree(ROOT, ignore_errors=True)
(ROOT/"store").mkdir(parents=True)
def add(path):
    p = pathlib.Path(path); h = hashlib.md5(p.read_bytes()).hexdigest()
    shutil.copy(p, ROOT/"store"/h)                       # warehouse: file stored under its hash
    pointer = {"path": p.name, "md5": h, "size": p.stat().st_size}
    pathlib.Path(str(p)+".ptr.json").write_text(json.dumps(pointer))   # the label
    return pointer
def checkout(ptr_file, dest):
    ptr = json.loads(pathlib.Path(ptr_file).read_text()); shutil.copy(ROOT/"store"/ptr["md5"], dest)
data = ROOT/"train.csv"
data.write_text("age,churn\n25,0\n40,1\n33,0\n")
p1 = add(data); print("v1 pointer:", p1)
ptr1 = ROOT/"train.csv.ptr.json"; shutil.copy(ptr1, ROOT/"v1.ptr.json")   # "git commit" of the label
data.write_text("age,churn\n25,0\n40,1\n33,0\n51,1\n")
p2 = add(data); print("v2 pointer:", p2)
checkout(ROOT/"v1.ptr.json", ROOT/"restored.csv")
print("restored v1 content:", repr((ROOT/"restored.csv").read_text()))
print("store objects:", sorted(x.name[:8] for x in (ROOT/"store").iterdir()))

## 4.2 Table-Level Versioning: Delta, Iceberg, lakeFS

In [ ]:
import pandas as pd, hashlib
v1 = pd.DataFrame({"id":[1,2,3,4],"age":[25,40,33,51],"churn":[0,1,0,1]})
v2 = pd.DataFrame({"id":[1,2,3,5],"age":[25,41,33,29],"churn":[0,1,0,0]})
def rowhash(df): return df.set_index("id").apply(lambda r: hashlib.md5(str(tuple(r)).encode()).hexdigest()[:6], axis=1)
a, b = rowhash(v1), rowhash(v2)
added = sorted(set(b.index)-set(a.index)); removed = sorted(set(a.index)-set(b.index))
changed = sorted(i for i in set(a.index)&set(b.index) if a[i]!=b[i])
print("added:", added, "removed:", removed, "changed:", changed)
print("rows v1:", len(v1), "rows v2:", len(v2))
print("positive rate v1:", v1.churn.mean(), "v2:", v2.churn.mean())

## 4.3 Datasheets and Model Cards

## 4.4 Versioning from Zero: What Exactly Are You Pinning?

Sections 4.1 to 4.3 showed three tools. Before going deeper into each, we need one clear mental model, because every problem in this chapter is a variation on it. Read this section slowly. Everything later depends on it.

In [ ]:
import hashlib, json

def short(b):
    return hashlib.sha256(b).hexdigest()[:10]

def make_record(commit, data_bytes, config, lock_bytes):
    rec = {
        "code": commit,
        "data": short(data_bytes),
        "config": short(json.dumps(config, sort_keys=True).encode()),
        "env": short(lock_bytes),
    }
    rec["run"] = short(json.dumps(rec, sort_keys=True).encode())
    return rec

data = b"age,churn\n25,0\n40,1\n33,0\n"
lock = b"numpy==1.26.4\nscikit-learn==1.4.2\n"
a = make_record("a1b2c3d", data, {"model": "logreg", "C": 1.0, "seed": 42}, lock)
b = make_record("a1b2c3d", data, {"model": "logreg", "C": 0.5, "seed": 42}, lock)
c = make_record("a1b2c3d", data + b"51,1\n", {"model": "logreg", "C": 1.0, "seed": 42}, lock)

print(json.dumps(a, indent=1))
for name, other in [("b", b), ("c", c)]:
    changed = [k for k in ("code", "data", "config", "env") if a[k] != other[k]]
    print(f"a vs {name}: changed pins = {changed}, same run id? {a['run'] == other['run']}")

## 4.5 Hashes and Content Addressing in Depth

Section 4.1 used `md5` without explaining what makes a hash trustworthy. Here we build the properties from experiments, then solve the practical problems: hashing a file too big for memory, and hashing a whole folder.

In [ ]:
import hashlib

for s in [b"", b"abc", b"abd"]:
    print(s, hashlib.md5(s).hexdigest(), hashlib.sha256(s).hexdigest()[:16])

x = int(hashlib.sha256(b"abc").hexdigest(), 16)
y = int(hashlib.sha256(b"abd").hexdigest(), 16)
print("bits that differ between abc and abd:", bin(x ^ y).count("1"), "of 256")
print("md5 hex length:", len(hashlib.md5(b"abc").hexdigest()), "sha256 hex length:", len(hashlib.sha256(b"abc").hexdigest()))

In [ ]:
import hashlib, pathlib, numpy as np

path = pathlib.Path("/tmp/lab/big.bin")
path.parent.mkdir(parents=True, exist_ok=True)
path.write_bytes(np.random.default_rng(0).bytes(20 * 1024 * 1024))   # 20 MB

def sha_whole(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

def sha_stream(p, chunk=1024 * 1024):
    h = hashlib.sha256()
    n = 0
    with open(p, "rb") as f:
        while True:
            block = f.read(chunk)
            if not block:
                break
            h.update(block)
            n += 1
    return h.hexdigest(), n

whole = sha_whole(path)
streamed, blocks = sha_stream(path)
print("size MB:", path.stat().st_size / 1024 / 1024)
print("blocks read:", blocks)
print("same hash:", whole == streamed)
print("first 12:", streamed[:12])

In [ ]:
import hashlib, pathlib, shutil

root = pathlib.Path("/tmp/lab/dirdemo")
shutil.rmtree(root, ignore_errors=True)
(root / "images").mkdir(parents=True)
(root / "labels.csv").write_text("img,y\na.png,0\nb.png,1\n")
(root / "images" / "a.png").write_bytes(b"AAAA")
(root / "images" / "b.png").write_bytes(b"BBBB")

def dir_hash(folder):
    lines = []
    for p in sorted(folder.rglob("*")):
        if p.is_file():
            rel = p.relative_to(folder).as_posix()
            lines.append(rel + ":" + hashlib.sha256(p.read_bytes()).hexdigest())
    return hashlib.sha256("\n".join(lines).encode()).hexdigest()[:12]

h0 = dir_hash(root)
print("original      :", h0)
print("same again    :", dir_hash(root) == h0)

(root / "images" / "b.png").rename(root / "images" / "c.png")
h1 = dir_hash(root)
print("renamed b->c  :", h1 != h0)

(root / "images" / "c.png").rename(root / "images" / "b.png")
print("renamed back  :", dir_hash(root) == h0)

(root / "images" / "b.png").write_bytes(b"BBBX")
print("one byte edit :", dir_hash(root) != h0)

(root / "empty.txt").write_bytes(b"")
print("empty file add:", dir_hash(root) != h0)

In [ ]:
def collision_probability(n_objects, bits):
    # birthday bound: p is about n^2 / 2^(bits+1)
    return n_objects ** 2 / 2 ** (bits + 1)

for n in (10 ** 6, 10 ** 9, 10 ** 12):
    print(f"{n:>16,d} objects  md5(128 bit): {collision_probability(n, 128):.2e}   sha256: {collision_probability(n, 256):.2e}")

## 4.6 Chunking and Deduplication: Storing Only What Changed

Section 4.1 stored each version as a whole file. If a 20 GB file changes in 2% of its content, whole-file versioning stores another 20 GB. This section shows how to store only the changed pieces, and the trap that breaks the obvious method.

In [ ]:
dataset_gb = 20
experiments = 10
variants = [("raw copy per experiment", dataset_gb * experiments),
            ("content-addressed, 1 distinct dataset", dataset_gb),
            ("content-addressed, 3 distinct datasets", dataset_gb * 3)]
for name, gb in variants:
    print(f"{name:42s} {gb:4d} GB")

# chunk-level reuse: a 20 GB file where 2% of 4 MB chunks change between versions
chunk_mb = 4
chunks = dataset_gb * 1024 // chunk_mb
changed = int(chunks * 0.02)
print("chunks per version:", chunks, " changed:", changed)
print("whole-file versioning, 5 versions:", 5 * dataset_gb, "GB")
print("chunk-level versioning, 5 versions:", round(dataset_gb + 4 * changed * chunk_mb / 1024, 2), "GB")

In [ ]:
import hashlib, numpy as np

data = np.random.default_rng(1).bytes(400_000)
edited = b"X" + data                      # insert ONE byte at the very start

def fixed_chunks(b, size=4096):
    return [b[i:i + size] for i in range(0, len(b), size)]

rng = np.random.default_rng(7)
GEAR = [int(v) for v in rng.integers(0, 2**32, 256, dtype=np.uint64)]

def cdc_chunks(b, mask=0x0FFF, lo=1024, hi=16384):
    chunks, start, h = [], 0, 0
    for i, byte in enumerate(b):
        h = ((h << 1) + GEAR[byte]) & 0xFFFFFFFF
        size = i - start + 1
        if (size >= lo and (h & mask) == 0) or size >= hi:
            chunks.append(b[start:i + 1])
            start, h = i + 1, 0
    if start < len(b):
        chunks.append(b[start:])
    return chunks

def ids(chunks):
    return [hashlib.md5(c).hexdigest() for c in chunks]

for name, fn in [("fixed 4 KB", fixed_chunks), ("content-defined", cdc_chunks)]:
    old, new = ids(fn(data)), ids(fn(edited))
    reused = len(set(old) & set(new))
    print(f"{name:16s} chunks old={len(old):3d} new={len(new):3d} reused={reused:3d} "
          f"-> new bytes to upload ~ {100 * (1 - reused / len(new)):.0f}% of chunks")

## 4.7 Pipelines That Rebuild Only What Changed (the DVC Pipeline Idea)

Versioning one file is the easy part. Real projects are chains: download, clean, build features, train, report. When one input changes, you want to rerun only the stages that depend on it. This is the second half of what DVC does, and the same idea powers Make, Bazel and most workflow tools.

In [ ]:
import hashlib, json, pathlib, shutil

ROOT = pathlib.Path("/tmp/lab/minidvc")
shutil.rmtree(ROOT, ignore_errors=True)
(ROOT / "cache").mkdir(parents=True)

def h(p):
    return hashlib.md5(pathlib.Path(p).read_bytes()).hexdigest()

def prep():
    lines = (ROOT / "raw.csv").read_text().splitlines()[1:]
    rows = [l for l in lines if l.strip()]
    (ROOT / "clean.csv").write_text("age,churn\n" + "\n".join(rows) + "\n")

def train():
    ps = json.loads((ROOT / "params.json").read_text())
    rows = [l.split(",") for l in (ROOT / "clean.csv").read_text().splitlines()[1:]]
    rate = sum(int(r[1]) for r in rows) / len(rows)
    (ROOT / "model.json").write_text(json.dumps({"rate": round(rate, 3), "C": ps["C"]}))

STAGES = {
    "prep":  {"deps": ["raw.csv"], "outs": ["clean.csv"], "fn": prep},
    "train": {"deps": ["clean.csv", "params.json"], "outs": ["model.json"], "fn": train},
}
lock = {}

def repro():
    for name, st in STAGES.items():
        cur = {d: h(ROOT / d) for d in st["deps"]}
        done = all((ROOT / o).exists() for o in st["outs"])
        if lock.get(name, {}).get("deps") == cur and done:
            print(f"  {name}: skipped (inputs unchanged)")
            continue
        st["fn"]()
        lock[name] = {"deps": cur}
        for o in st["outs"]:
            shutil.copy(ROOT / o, ROOT / "cache" / h(ROOT / o))
        print(f"  {name}: ran")

(ROOT / "raw.csv").write_text("age,churn\n25,0\n40,1\n33,0\n51,1\n")
(ROOT / "params.json").write_text(json.dumps({"C": 1.0}))
print("run 1 (first time)"); repro()
print("run 2 (nothing changed)"); repro()
(ROOT / "params.json").write_text(json.dumps({"C": 0.5}))
print("run 3 (only params changed)"); repro()
(ROOT / "raw.csv").write_text("age,churn\n25,0\n40,1\n33,0\n51,1\n\n")   # add a blank line
print("run 4 (raw changed, but cleaning removes the blank line)"); repro()
(ROOT / "raw.csv").write_text("age,churn\n25,0\n40,1\n33,0\n51,1\n29,1\n")
print("run 5 (a real new row)"); repro()
print("model:", (ROOT / "model.json").read_text())
print("cache objects:", len(list((ROOT / "cache").iterdir())))

In [ ]:
stages = {
    "download": {"deps": [], "outs": ["raw.csv"]},
    "clean":    {"deps": ["raw.csv"], "outs": ["clean.csv"]},
    "features": {"deps": ["clean.csv", "feat_params.yaml"], "outs": ["feats.parquet"]},
    "train":    {"deps": ["feats.parquet", "train_params.yaml"], "outs": ["model.pkl"]},
    "report":   {"deps": ["model.pkl"], "outs": ["report.md"]},
}

def stages_to_rerun(changed):
    changed, rerun = set(changed), []
    for name, st in stages.items():                  # dict is in dependency order
        if changed & set(st["deps"]):
            rerun.append(name)
            changed |= set(st["outs"])               # its outputs now count as changed
    return rerun

for ch in (["raw.csv"], ["train_params.yaml"], ["feat_params.yaml"], []):
    print(f"changed {str(ch):22s} -> rerun {stages_to_rerun(ch)}")

## 4.8 Table Versioning Internals: Logs, Snapshots, Retention and Cost

Section 4.2 introduced `VERSION AS OF`. Here we open the box. Understanding the log is what lets you answer "why did time travel fail?" and "why is my storage bill growing?".

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE files (file TEXT, id INT, age INT, churn INT);
CREATE TABLE log   (version INT, op TEXT, file TEXT, day TEXT);
""")

def commit(version, day, add_rows=None, add_file=None, remove=()):
    for f in remove:
        db.execute("INSERT INTO log VALUES (?, 'remove', ?, ?)", (version, f, day))
    if add_file:
        db.executemany("INSERT INTO files VALUES (?,?,?,?)", [(add_file, *r) for r in add_rows])
        db.execute("INSERT INTO log VALUES (?, 'add', ?, ?)", (version, add_file, day))

commit(0, "2026-01-05", [(1, 25, 0), (2, 40, 1), (3, 33, 0), (4, 51, 1)], "f0")
commit(1, "2026-01-20", [(1, 25, 0), (2, 41, 1), (3, 33, 0), (4, 51, 1)], "f1", remove=["f0"])
commit(2, "2026-02-10", [(5, 29, 0)], "f2")

AS_OF = """SELECT id, age, churn FROM files WHERE file IN
  (SELECT file FROM log WHERE op='add' AND version<=:v
   EXCEPT SELECT file FROM log WHERE op='remove' AND version<=:v) ORDER BY id"""

for v in (0, 1, 2):
    print(f"version {v}:", db.execute(AS_OF, {"v": v}).fetchall())

print("history:", db.execute("SELECT version, op, file, day FROM log ORDER BY version, op").fetchall())

# VACUUM: physically delete files that no current-snapshot version needs
live = {r[0] for r in db.execute("SELECT file FROM log WHERE op='add' EXCEPT SELECT file FROM log WHERE op='remove'")}
db.execute("DELETE FROM files WHERE file NOT IN (%s)" % ",".join("?" * len(live)), tuple(live))
print("after VACUUM, version 0:", db.execute(AS_OF, {"v": 0}).fetchall())
print("after VACUUM, version 2:", db.execute(AS_OF, {"v": 2}).fetchall())
print("files still on disk:", sorted(r[0] for r in db.execute("SELECT DISTINCT file FROM files")))

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE plan_history (customer INT, plan TEXT, valid_from TEXT, valid_to TEXT);
INSERT INTO plan_history VALUES
 (1, 'basic',   '2025-01-01', '2025-06-01'),
 (1, 'premium', '2025-06-01', '9999-12-31'),
 (2, 'basic',   '2025-01-01', '9999-12-31'),
 (3, 'basic',   '2025-01-01', '2025-03-01'),
 (3, 'premium', '2025-03-01', '2025-09-01'),
 (3, 'basic',   '2025-09-01', '9999-12-31');
CREATE TABLE labels (customer INT, label_date TEXT, churned INT);
INSERT INTO labels VALUES (1,'2025-05-15',1),(2,'2025-05-15',0),(3,'2025-05-15',0),(1,'2025-10-01',0),(3,'2025-10-01',1);
""")

q = """SELECT l.customer, l.label_date, p.plan, l.churned
FROM labels l JOIN plan_history p
  ON p.customer = l.customer AND l.label_date >= p.valid_from AND l.label_date < p.valid_to
ORDER BY l.label_date, l.customer"""
for row in db.execute(q):
    print(row)

wrong = db.execute("""SELECT l.customer, l.label_date, p.plan FROM labels l JOIN plan_history p
  ON p.customer = l.customer AND p.valid_to = '9999-12-31' ORDER BY l.label_date, l.customer""").fetchall()
print("leaky join (uses the CURRENT plan):", wrong)

In [ ]:
rows_per_file = 10_000
table_gb = 100
daily_change = 0.02            # fraction of rows edited each day
price = 0.023                  # illustrative USD per GB-month for object storage

p_untouched = (1 - daily_change) ** rows_per_file
touched_random = 1 - p_untouched
print("fraction of files rewritten per day if edits are spread randomly:", round(touched_random, 6))
touched_clustered = daily_change
print("fraction rewritten if edits are clustered by date (one partition):", touched_clustered)

for label, frac in [("random edits", touched_random), ("clustered edits", touched_clustered)]:
    for days in (7, 30, 90):
        extra_gb = table_gb * frac * days            # each day's rewrite is kept for `days` days
        total = table_gb + extra_gb
        print(f"{label:16s} retain {days:2d} d: {total:8.0f} GB  ${total * price:8.2f}/month")

## 4.9 Comparing Data Versions: Schema, Rows, Distributions

When a retrained model is worse, the cheapest first test is: did the data change, and how? A good data diff answers in seconds. Build it in three layers, from cheap to expensive.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(0)
n = 1000
v1 = pd.DataFrame({
    "id": np.arange(n),
    "age": rng.integers(18, 80, n),
    "plan": rng.choice(["basic", "premium"], n, p=[0.7, 0.3]),
})
v1["churn"] = (rng.random(n) < 0.20).astype(int)

v2 = v1.copy()
v2.loc[v2.sample(100, random_state=1).index, "age"] *= 12      # bug: some ages became months
v2 = pd.concat([v2, v2.iloc[:30]])                              # bug: 30 duplicated rows
v2.loc[v2.sample(60, random_state=2).index, "plan"] = "trial"   # new category
v2["region"] = "EU"                                             # new column

def diff_report(a, b, key, label):
    out = {}
    out["columns added"] = sorted(set(b.columns) - set(a.columns))
    out["columns removed"] = sorted(set(a.columns) - set(b.columns))
    out["rows"] = (len(a), len(b))
    out["duplicate keys"] = (int(a[key].duplicated().sum()), int(b[key].duplicated().sum()))
    out["label rate"] = (round(float(a[label].mean()), 3), round(float(b[label].mean()), 3))
    common = [c for c in a.columns if c in b.columns and c not in (key, label)]
    for c in common:
        if pd.api.types.is_numeric_dtype(a[c]):
            shift = (b[c].mean() - a[c].mean()) / a[c].std()
            out[f"{c} mean shift (in std of v1)"] = round(shift, 2)
        else:
            out[f"{c} new values"] = sorted(set(b[c]) - set(a[c]))
    return out

for k, v in diff_report(v1, v2, "id", "churn").items():
    print(f"{k:34s} {v}")

## 4.10 Model Versioning, Lineage and Fair Comparison

Data is half of the story. The other half is the model artifact: how to name it, prove which pins made it, tell two versions apart, compare them fairly, and load it safely.

In [ ]:
import hashlib, numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=500, n_features=5, random_state=0)

def fingerprint(model):
    blob = np.round(model.coef_, 10).tobytes() + np.round(model.intercept_, 10).tobytes()
    return hashlib.sha256(blob).hexdigest()[:10]

m1 = LogisticRegression(C=1.0).fit(X, y)
m2 = LogisticRegression(C=1.0).fit(X, y)
m3 = LogisticRegression(C=0.1).fit(X, y)
m4 = LogisticRegression(C=1.0).fit(X[:-1], y[:-1])        # one training row removed

print("train twice, same data/config, same fingerprint:", fingerprint(m1) == fingerprint(m2))
print("different C, same fingerprint:                  ", fingerprint(m1) == fingerprint(m3))
print("one row fewer, same fingerprint:                ", fingerprint(m1) == fingerprint(m4))
print("coefficient change from dropping 1 row of 500:  ", float(np.abs(m1.coef_ - m4.coef_).max()).__round__(6))

In [ ]:
import hashlib, numpy as np

vals = np.random.default_rng(0).normal(size=100_000).astype(np.float32)
a = vals.sum()                                   # one order of addition
b = vals[::-1].sum()                             # same numbers, reverse order
print("sums equal exactly?", a == b, "| difference:", float(a - b))
print("raw bytes hash equal?", hashlib.md5(np.float32(a).tobytes()).hexdigest() == hashlib.md5(np.float32(b).tobytes()).hexdigest())
print("rounded to 4 decimals equal?", round(float(a), 4) == round(float(b), 4))

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=2000, n_features=8, n_informative=4, flip_y=0.05, random_state=3)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)

v1 = LogisticRegression(C=1.0, max_iter=1000).fit(Xtr[:600], ytr[:600])
v2 = LogisticRegression(C=1.0, max_iter=1000).fit(Xtr, ytr)

p1, p2 = v1.predict(Xte), v2.predict(Xte)
print("accuracy v1:", round((p1 == yte).mean(), 4), " v2:", round((p2 == yte).mean(), 4))
print("predictions that differ:", int((p1 != p2).sum()), "of", len(yte), "=", round((p1 != p2).mean(), 4))
fixed = int(((p1 != yte) & (p2 == yte)).sum()); broken = int(((p1 == yte) & (p2 != yte)).sum())
print("v2 fixed", fixed, "v1 mistakes, but broke", broken, "v1 successes")
print("net:", fixed - broken, "-> accuracy gain", round((fixed - broken) / len(yte), 4))

In [ ]:
groups = ["A", "B"]
v1 = {"A": (800, 0.90), "B": (200, 0.62)}      # (test rows, accuracy)
v2 = {"A": (300, 0.91), "B": (700, 0.64)}      # new test set has a different mix

def overall(d):
    n = sum(r for r, _ in d.values())
    return sum(r * a for r, a in d.values()) / n

for g in groups:
    print(f"group {g}: v1 acc {v1[g][1]:.2f} -> v2 acc {v2[g][1]:.2f}   (rows {v1[g][0]} -> {v2[g][0]})")
print("overall v1:", round(overall(v1), 3), " overall v2:", round(overall(v2), 3))

# fair comparison: score v2's per-group accuracies on v1's mix
v2_on_v1_mix = {g: (v1[g][0], v2[g][1]) for g in groups}
print("v2 re-weighted to v1's mix:", round(overall(v2_on_v1_mix), 3))

In [ ]:
def bump(version, change):
    major, minor, patch = map(int, version.split("."))
    if change in ("input_schema", "output_schema", "label_definition"):
        return f"{major + 1}.0.0"
    if change in ("new_training_data", "new_features", "new_algorithm"):
        return f"{major}.{minor + 1}.0"
    if change in ("retrain_same_data_new_seed", "bug_fix_preprocessing", "doc_only"):
        return f"{major}.{minor}.{patch + 1}"
    raise ValueError("unknown change: " + change)

v = "2.3.1"
for ch in ["new_training_data", "input_schema", "doc_only", "new_algorithm"]:
    print(f"{v} + {ch:18s} -> {bump(v, ch)}")
try:
    bump(v, "magic")
except ValueError as e:
    print("error:", e)

In [ ]:
import pickle

class Evil:
    def __reduce__(self):
        # harmless demo: a real attacker would call os.system or similar here
        return (print, ("code ran just because the file was LOADED",))

blob = pickle.dumps(Evil())
print("bytes in file:", len(blob))
obj = pickle.loads(blob)           # nobody called anything: loading was enough
print("loaded object:", obj)

## 4.11 How to Solve Any Data and Model Versioning Problem

This section is a playbook. Most real questions, in work or in an interview, are one of eight patterns. First learn to recognise the pattern, then apply its fixed recipe, then check the answer. Use Figure 4.8 as the starting map.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE runs (run_id TEXT, code TEXT, data_hash TEXT, config_hash TEXT, env_hash TEXT, auc REAL);
CREATE TABLE models (name TEXT, version INT, alias TEXT, run_id TEXT);
INSERT INTO runs VALUES
 ('r1','c10','dA','k1','e1',0.81),
 ('r2','c11','dA','k2','e1',0.83),
 ('r3','c11','dB','k2','e1',0.86),
 ('r4','c12','dB','k3','e2',0.85);
INSERT INTO models VALUES
 ('churn',1,'archived','r1'),('churn',2,'champion','r3'),('churn',3,'challenger','r4'),('fraud',1,'champion','r2');
""")

print("Q1 which data trained the production churn model?")
print(db.execute("""SELECT m.version, r.data_hash, r.code FROM models m JOIN runs r USING(run_id)
                    WHERE m.name='churn' AND m.alias='champion'""").fetchall())

print("Q2 data dA was found to be corrupt: which models are affected?")
print(db.execute("""SELECT m.name, m.version, m.alias FROM models m JOIN runs r USING(run_id)
                    WHERE r.data_hash='dA' ORDER BY m.name, m.version""").fetchall())

print("Q3 same code c11, different data: AUC effect of the data")
print(db.execute("""SELECT data_hash, auc FROM runs WHERE code='c11' ORDER BY data_hash""").fetchall())

In [ ]:
import hashlib, sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE t (version INT, id INT, age INT, churn INT)")
db.executemany("INSERT INTO t VALUES (?,?,?,?)", [
    (0, 1, 25, 0), (0, 2, 40, 1), (0, 3, 33, 0),
    (1, 1, 25, 0), (1, 2, 41, 1), (1, 3, 33, 0), (1, 4, 29, 1),
])

def dataset_hash(version):
    rows = db.execute("SELECT id, age, churn FROM t WHERE version=? ORDER BY id", (version,)).fetchall()
    return hashlib.sha256(repr(rows).encode()).hexdigest()[:12]

recorded = {"model_v7": {"table_version": 1, "data_hash": dataset_hash(1)}}   # saved at training time
print("recorded at training time:", recorded["model_v7"])

def verify(name):
    rec = recorded[name]
    now = dataset_hash(rec["table_version"])
    return "OK, identical data" if now == rec["data_hash"] else f"MISMATCH (now {now})"

print("verify today:", verify("model_v7"))
db.execute("UPDATE t SET age = 42 WHERE version=1 AND id=2")     # someone edits history in place
print("after silent edit:", verify("model_v7"))

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=3000, n_features=10, n_informative=5, random_state=11)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

# data v2: a "cleaning" step that accidentally drops 80% of the positive rows
rng = np.random.default_rng(5)
keep = (ytr == 0) | (rng.random(len(ytr)) < 0.2)
Xtr_v2, ytr_v2 = Xtr[keep], ytr[keep]
print("positive rate train v1:", round(float(ytr.mean()), 3), " v2:", round(float(ytr_v2.mean()), 3))

def score(C, data):
    Xa, ya = (Xtr, ytr) if data == "v1" else (Xtr_v2, ytr_v2)
    model = LogisticRegression(C=C, max_iter=200).fit(Xa, ya)
    return round(model.score(Xte, yte), 4)         # always evaluated on the SAME test set

grid = pd.DataFrame({d: {f"C={c}": score(c, d) for c in (1.0, 0.01)} for d in ("v1", "v2")})
grid.columns = ["data v1", "data v2"]
print(grid)
print("effect of changing only C (data v1):  ", round(grid.loc["C=0.01", "data v1"] - grid.loc["C=1.0", "data v1"], 4))
print("effect of changing only data (C=1.0): ", round(grid.loc["C=1.0", "data v2"] - grid.loc["C=1.0", "data v1"], 4))

In [ ]:
import pandas as pd

def classify_change(a, b, key):
    if list(a.columns) != list(b.columns):
        return "schema change: columns differ"
    ka, kb = set(a[key]), set(b[key])
    if ka != kb:
        return f"row set change: added {sorted(kb - ka)}, removed {sorted(ka - kb)}"
    merged = a.merge(b, on=key, suffixes=("_old", "_new"))
    cols = [c for c in a.columns if c != key]
    changed = [c for c in cols if (merged[c + "_old"] != merged[c + "_new"]).any()]
    return f"value change in columns {changed}" if changed else "identical"

base = pd.DataFrame({"id": [1, 2, 3], "age": [25, 40, 33], "churn": [0, 1, 0]})
cases = {
    "same":       base.copy(),
    "new column": base.assign(region="EU"),
    "new row":    pd.concat([base, pd.DataFrame({"id": [4], "age": [29], "churn": [1]})]),
    "edit":       base.assign(age=[25, 41, 33]),
}
for name, other in cases.items():
    print(f"{name:11s} -> {classify_change(base, other, 'id')}")

In [ ]:
def where_to_store(size_mb, changes_per_month):
    if size_mb <= 1:
        return "git (small text)"
    if size_mb <= 100 and changes_per_month <= 1:
        return "git-lfs or DVC (large but rarely changes)"
    if changes_per_month > 30:
        return "table format (Delta/Iceberg) with time travel"
    return "DVC / content-addressed store with pointer in git"

for args in [(0.2, 5), (50, 1), (3000, 4), (20000, 200)]:
    print(args, "->", where_to_store(*args))

# why big files hurt: every version stays in history
file_gb, versions, speed_mb_s = 2, 12, 50
history_gb = file_gb * versions
print("clone size:", history_gb, "GB; at", speed_mb_s, "MB/s takes", round(history_gb * 1024 / speed_mb_s / 60, 1), "minutes")

In [ ]:
from datetime import date, timedelta

def reproducible(train_day, today, retention_days, vacuum_every_days=7):
    # a version is deletable once it is older than retention; vacuum runs on a schedule
    age = (today - train_day).days
    worst_case_gone = age > retention_days
    sure_available = age <= retention_days
    return age, sure_available, worst_case_gone

today = date(2026, 3, 10)
for retention in (7, 30, 90):
    age, ok, gone = reproducible(date(2026, 1, 15), today, retention)
    print(f"retention {retention:3d} d: age={age} d, guaranteed={ok}, may be deleted={gone}")

need_days = (today - date(2025, 9, 1)).days
print("to reproduce 1 Sep 2025 you need retention of at least", need_days, "days")
print("latest safe training date for a 30-day window:", today - timedelta(days=30))

In [ ]:
REQUIRED = ["intended_use", "not_for", "training_data", "eval_data", "metrics", "per_group", "limits", "owner", "review_date"]

def lint(card):
    problems = [f"missing: {k}" for k in REQUIRED if not card.get(k)]
    if card.get("per_group"):
        accs = [g["acc"] for g in card["per_group"].values()]
        if max(accs) - min(accs) > 0.10:
            problems.append(f"group gap {max(accs) - min(accs):.2f} > 0.10: needs a mitigation note")
        n = sum(g["n"] for g in card["per_group"].values())
        w = sum(g["n"] * g["acc"] for g in card["per_group"].values()) / n
        if abs(w - card["metrics"]["acc"]) > 0.005:
            problems.append(f"overall acc {card['metrics']['acc']} does not match weighted groups {w:.3f}")
    return problems or ["OK"]

card = {
    "intended_use": "rank customers for retention calls", "not_for": "pricing, credit",
    "training_data": "churn table md5 9e2f9675", "eval_data": "450 held-out rows",
    "metrics": {"acc": 0.83}, "limits": "one region", "owner": "ml-platform",
    "per_group": {"A": {"n": 800, "acc": 0.90}, "B": {"n": 200, "acc": 0.62}},
}
for p in lint(card):
    print(p)

## 4.12 Going Deeper: Internals, Edge Cases, Performance and What Interviewers Probe

This part is for when the basics feel comfortable. Skip it on first reading and return later. It is the knowledge that separates "I used DVC once" from "I can design and debug a versioning system".

In [ ]:
import hashlib, unicodedata

name_nfc = unicodedata.normalize("NFC", "café.csv")      # e-acute as one character
name_nfd = unicodedata.normalize("NFD", "café.csv")      # e + combining accent
print("look the same when printed:", name_nfc, name_nfd)
print("equal as strings?", name_nfc == name_nfd)
print("lengths:", len(name_nfc), len(name_nfd))
for label, n in [("NFC", name_nfc), ("NFD", name_nfd)]:
    print(label, hashlib.sha256(n.encode("utf-8")).hexdigest()[:12])
print("hash after normalising both to NFC equal?",
      hashlib.sha256(unicodedata.normalize("NFC", name_nfd).encode()).hexdigest() ==
      hashlib.sha256(name_nfc.encode()).hexdigest())

In [ ]:
import gzip, hashlib

payload = b"age,churn\n25,0\n40,1\n"
a = gzip.compress(payload, mtime=1000)
b = gzip.compress(payload, mtime=2000)
c = gzip.compress(payload, mtime=0)
d = gzip.compress(payload, mtime=0)
print("same content, different timestamp -> same bytes?", a == b)
print("same content, fixed timestamp     -> same bytes?", c == d)
print("decompressed content equal?", gzip.decompress(a) == gzip.decompress(b))
print("hashes:", hashlib.md5(a).hexdigest()[:8], hashlib.md5(b).hexdigest()[:8])

## 4.13 Mixed-Practice Drill and Self-Test

The drill mixes patterns. For each problem, first name the pattern (Section 4.11), then solve it on paper, then read the answer.

## 4.14 Where This Connects: Before and After

## Chapter recap

## Practice set

Problem 4.1 What is stored in a DVC pointer file? **Answer:** The hash, size and path of the real data.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch04.html